# Banyan City — free Wan T2V rendering on Kaggle
Renders a node's `shots.md` prompts into per-beat clips with **open Apache-2.0
[Wan 2.1 T2V 1.3B](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers) weights** on Kaggle's
free GPU quota (30 h/week) — the tree's permanent $0 rendering floor, reproducible by any citizen
(**compute-as-watering**, see `WATERING.md`).

**Setup:** Kaggle → New Notebook → File → Import Notebook → this file. Settings: Accelerator = GPU
(T4/P100), Internet = ON, phone-verified account. Then *Run All*. ~20-45 min per 5s clip; a 5-beat
episode ≈ one afternoon of queue. Output: `/kaggle/working/clips.zip` → feed to
`pipeline/render_t3.py <genome> <node> --clips <dir>`.

Provenance: every clip gets a `meta.yaml` (§7.2). Output is 480×832 (9:16) 5s — the current best a
free T4 does; the season's canon quality bar is decided by the founder on material (R4/D8).

In [ ]:
# ---- config: what to render ----------------------------------------------
GENOME = "sapling"
NODE   = "002b"        # any node id with a shots.md
BEATS  = None          # e.g. [1, 3] or None for all beats without status ✅
SEED   = 20260719      # fixed base seed: beat N renders with SEED + N (reproducible)
STEPS  = 40            # 30 = faster/rougher, 50 = slower/cleaner
REPO_URL = "https://github.com/olegmlkvorg/banyan-city

In [ ]:
# ---- setup: deps + repo (canon prompts come from shots.md, not a paste) ---
%pip -q install "diffusers>=0.33" transformers accelerate ftfy imageio imageio-ffmpeg pyyaml
import subprocess, sys, pathlib
if not pathlib.Path("banyan-city").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
sys.path.insert(0, "banyan-city/pipeline")
from generate_shots import parse_shots
import yaml
node_dirs = list(pathlib.Path(f"banyan-city/genomes/{GENOME}/nodes").glob("*/"))
node_dir = next(d for d in node_dirs if d.name.startswith(NODE))
shots = parse_shots((node_dir / "shots.md").read_text())
todo = [s for s in shots if (BEATS is None and not s["done"]) or (BEATS and s["num"] in BEATS)]
print(f"{len(todo)} beat(s) to render for {node_dir.name}:")
for s in todo: print(f"  {s['num']:02d} {s['slug']}")

In [ ]:
# ---- model: Wan 2.1 T2V 1.3B (Apache-2.0), fp16, offloaded for 16GB GPUs ---
import torch
from diffusers import WanPipeline
from diffusers.utils import export_to_video
pipe = WanPipeline.from_pretrained("Wan-AI/Wan2.1-T2V-1.3B-Diffusers", torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
print("pipeline ready")

In [ ]:
# ---- generate: 480x832 (9:16), 81 frames @ 16fps = ~5s per beat ------------
from datetime import date
out = pathlib.Path("/kaggle/working/clips"); out.mkdir(parents=True, exist_ok=True)
for s in todo:
    dest = out / f"{s['num']:02d}-{s['slug']}.mp4"
    if dest.exists():
        print(f"skip {dest.name} (exists)"); continue
    print(f"beat {s['num']:02d} ({s['slug']}) …", flush=True)
    g = torch.Generator(device="cpu").manual_seed(SEED + s["num"])
    frames = pipe(prompt=s["prompt"],
                  negative_prompt="photorealistic, 3d render, text, watermark, low quality, blurry",
                  height=832, width=480, num_frames=81,
                  num_inference_steps=STEPS, generator=g).frames[0]
    export_to_video(frames, str(dest), fps=16)
    dest.with_suffix(".meta.yaml").write_text(
        "# Shot provenance (\u00a77.2)\n" + yaml.safe_dump({
            "platform": "kaggle-free-gpu", "model": "Wan2.1-T2V-1.3B (Apache-2.0)",
            "shot_beat": s["num"], "prompt": s["prompt"], "seed": SEED + s["num"],
            "steps": STEPS, "duration_s": 5, "aspect": "9:16 (480x832)",
            "cost_usd": 0.0, "date": date.today().isoformat(),
        }, sort_keys=False, allow_unicode=True))
    print(f"  \u2713 {dest.name}")

In [ ]:
# ---- pack for download ------------------------------------------------------
import shutil
shutil.make_archive("/kaggle/working/clips", "zip", "/kaggle/working/clips")
print("download clips.zip from the Output tab, then locally:")
print(f"  python3 pipeline/render_t3.py {GENOME} {NODE} --clips <unzipped-dir> --out /tmp/{NODE}-episode.mp4")